# Travel Agency Booking and Tourism Trend Analysis System
### Foundations of Data Science (25ITC31) - Practical Notebook
**Student Name:** Kishor K  
**Register Number:** 732925ITR051  
**Class:** 25IT3A  
**Department:** Information Technology  
**Institution:** Velalar College of Engineering and Technology, Erode - 12  
**Assigned Scenario:** Scenario 21 (Travel Agency Booking and Tourism Trend Analysis System)

## 1. Import Essential Libraries
In accordance with Unit 2 and Unit 3 of the syllabus, we import `pandas`, `numpy`, `matplotlib.pyplot`, `seaborn`, and the `sqlite3` database driver.

In [ ]:
import os
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted")
print("All data science libraries imported successfully!")

## 2. Database Extraction
We retrieve the relational dataset directly from the SQLite database (`travel_agency.db`) into Pandas DataFrames.

In [ ]:
db_path = os.path.join("..", "database", "travel_agency.db")
conn = sqlite3.connect(db_path)

customers_df = pd.read_sql_query("SELECT * FROM customers", conn)
packages_df = pd.read_sql_query("SELECT * FROM travel_packages", conn)
bookings_df = pd.read_sql_query("SELECT * FROM bookings", conn)
payments_df = pd.read_sql_query("SELECT * FROM payments", conn)
cancellations_df = pd.read_sql_query("SELECT * FROM cancellations", conn)

conn.close()
print(f"Loaded {len(customers_df)} customers, {len(packages_df)} packages, and {len(bookings_df)} bookings.")

## 3. Data Cleaning, Transformation & Feature Engineering (Unit 4)
- Convert date columns to `datetime64[ns]`.
- Derive `lead_time_days` between booking date and travel date.
- Extract month names and numbers for seasonal trend analysis.
- Merge tables into an analytical master dataframe.

In [ ]:
bookings_df['booking_date'] = pd.to_datetime(bookings_df['booking_date'])
bookings_df['travel_date'] = pd.to_datetime(bookings_df['travel_date'])
bookings_df['lead_time_days'] = (bookings_df['travel_date'] - bookings_df['booking_date']).dt.days
bookings_df['booking_month'] = bookings_df['booking_date'].dt.strftime('%B')
bookings_df['booking_month_num'] = bookings_df['booking_date'].dt.month

# Merge datasets
master_df = bookings_df.merge(packages_df, on='package_id', how='left')
master_df = master_df.merge(customers_df, on='customer_id', how='left', suffixes=('', '_cust'))
master_df = master_df.merge(payments_df[['booking_id', 'payment_method', 'payment_status']], on='booking_id', how='left')
master_df = master_df.merge(cancellations_df[['booking_id', 'reason', 'refund_amount']], on='booking_id', how='left')

master_df['refund_amount'] = master_df['refund_amount'].fillna(0.0)
master_df['reason'] = master_df['reason'].fillna("Active Booking")

master_df.head(5)

## 4. Descriptive Statistics (Unit 1 & Unit 5)
Compute central tendencies (mean, median, standard deviation) and business revenue totals.

In [ ]:
print("=== KEY PERFORMANCE INDICATORS ===")
print(f"Total Revenue Generated : Rs. {master_df['total_amount'].sum():,.2f}")
print(f"Average Booking Value   : Rs. {master_df['total_amount'].mean():,.2f}")
print(f"Average Party Size      : {master_df['num_travelers'].mean():.2f} travelers")
print(f"Average Lead Time       : {master_df['lead_time_days'].mean():.1f} days")
print(f"Cancellation Rate       : {(master_df['booking_status'] == 'CANCELLED').mean() * 100:.2f}%")

## 5. Visualizations & Pattern Discovery (Unit 5)
### Visualization 1: Destination Popularity Ranking

In [ ]:
plt.figure(figsize=(10, 5.5))
dest_demand = master_df[master_df['booking_status'] != 'CANCELLED'].groupby('destination')['num_travelers'].sum().reset_index()
dest_demand = dest_demand.sort_values(by='num_travelers', ascending=True)

bars = plt.barh(dest_demand['destination'], dest_demand['num_travelers'], color=sns.color_palette("viridis", len(dest_demand)), edgecolor='black')
for b in bars:
    w = b.get_width()
    plt.text(w + 0.3, b.get_y() + b.get_height()/2, f'{int(w)} travelers', va='center', fontweight='bold')

plt.title("Top Tourist Destinations by Confirmed Traveler Headcount", fontsize=13, fontweight='bold')
plt.xlabel("Total Travelers Booked")
plt.ylabel("Destination")
plt.show()

### Visualization 2: Monthly & Seasonal Tourism Demand Trajectory

In [ ]:
plt.figure(figsize=(10, 5))
monthly = master_df.groupby(['booking_month_num', 'booking_month'])['num_travelers'].sum().reset_index().sort_values('booking_month_num')
plt.plot(monthly['booking_month'], monthly['num_travelers'], marker='o', linewidth=2.5, color='#1f77b4', markersize=8)
plt.title("2026 Monthly Tourism Demand Trajectory", fontsize=13, fontweight='bold')
plt.xlabel("Month")
plt.ylabel("Traveler Headcount")
plt.show()

### Visualization 3: Revenue Contribution by Tour Category

In [ ]:
plt.figure(figsize=(7, 7))
cat_rev = master_df[master_df['booking_status'] != 'CANCELLED'].groupby('category')['total_amount'].sum()
plt.pie(cat_rev, labels=cat_rev.index, autopct='%1.1f%%', startangle=140, colors=sns.color_palette("Set2"),
        wedgeprops=dict(width=0.45, edgecolor='white', linewidth=2))
plt.title("Revenue Share by Package Category", fontsize=13, fontweight='bold')
plt.show()